# ResNet-18 CIFAKE: Standalone Full Training, Evaluation, and Robustness Notebook

This notebook is a **fully standalone** ResNet-18 pipeline for CIFAKE binary classification (**REAL=0**, **FAKE=1**).

It supports two workflows:
1. **FAST TEST MODE** (default): quick pipeline validation end-to-end.
2. **FULL TRAINING MODE**: full-data training/evaluation for final experiment results.

> After this notebook runs successfully in FAST TEST MODE, change to FULL TRAINING MODE to reproduce the full ResNet-18 experiment.

> This notebook does **not** require `git clone`, does **not** import `src.*`, and can train from scratch using only the CIFAKE dataset input.


## 1) Environment check

**What this does:** imports required libraries and checks runtime/device availability.

**Why needed:** confirms the environment is ready before data loading/training.

**Expected output:** successful imports plus CUDA/CPU and version info.


In [ ]:
import io
import os
import sys
import json
import math
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {DEVICE}")

# If an import above fails in Kaggle, install missing package with:
# !pip install <package_name>


## 2) Configuration: FAST_TEST vs FULL_TRAINING

**What this does:** defines all runtime settings and mode presets.

**Why needed:** keeps quick pipeline validation separate from the full experiment configuration.

**Expected output:** printed configuration summary plus a parameter comparison note.

> **Note:** FULL_TRAINING mode follows the original ResNet-18 experiment settings from the project repository. FAST_TEST mode is only for pipeline validation.


In [ ]:
DATA_DIR = Path("/kaggle/input/cifake-real-and-ai-generated-synthetic-images/CIFAKE")
OUTPUT_DIR = Path("outputs/resnet18_standalone")
FIG_DIR = OUTPUT_DIR / "figures"

IMG_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 42

MODE = "FAST_TEST"  # change to "FULL_TRAINING" after successful fast validation
FAST_DEV_RUN = MODE == "FAST_TEST"

if FAST_DEV_RUN:
    RUN_TRAINING = True
    RUN_CLEAN_EVAL = True
    RUN_ROBUSTNESS_EVAL = True
    RUN_VISUALIZATION = True
    RUN_INFERENCE_DEMO = True

    EPOCHS = 1
    BATCH_SIZE = 8
    NUM_WORKERS = 0
    MAX_TRAIN_EACH = 20
    MAX_TEST_EACH = 20
else:
    RUN_TRAINING = True
    RUN_CLEAN_EVAL = True
    RUN_ROBUSTNESS_EVAL = True
    RUN_VISUALIZATION = True
    RUN_INFERENCE_DEMO = True

    # Original project full experiment defaults
    EPOCHS = 10
    BATCH_SIZE = 128
    NUM_WORKERS = 2
    MAX_TRAIN_EACH = None
    MAX_TEST_EACH = None

JPEG_QUALITIES = [75, 50, 25]
GAUSSIAN_BLUR_SIGMAS = [1, 2, 3]
GAUSSIAN_NOISE_SIGMAS = [0.05, 0.10, 0.20]
TRAIN_VAL_SPLIT = 0.8
CHECKPOINT_SELECTION_METRIC = "val_f1"

for d in [OUTPUT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"MODE={MODE} | FAST_DEV_RUN={FAST_DEV_RUN}")
print(f"Device: {DEVICE}")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print("
FULL_TRAINING parameter summary (aligned with original project implementation):")
print("- EPOCHS: 10")
print("- BATCH_SIZE: 128")
print("- LEARNING_RATE: 1e-3")
print("- NUM_WORKERS: 2")
print("- MAX_TRAIN_EACH: None (full train split)")
print("- MAX_TEST_EACH: None (full test split)")
print("- Train/Val split: 80/20")
print("- Checkpoint selection metric: val_f1")
print("FULL_TRAINING parameters were aligned with the original project implementation.")


## 3) Utility functions


In [ ]:
@dataclass(frozen=True)
class Sample:
    image_path: Path
    label: int

LABEL_TO_INDEX = {"REAL": 0, "FAKE": 1}
INDEX_TO_LABEL = {0: "REAL", 1: "FAKE"}

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def ensure_dir(path: Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def list_image_files(directory: Path, max_files: int | None = None) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths = []
    count = 0
    for p in sorted(directory.iterdir()):
        if p.is_file() and p.suffix.lower() in exts:
            paths.append(p)
            count += 1
            if max_files is not None and count >= max_files:
                break
    return paths

def validate_cifake_dirs(data_dir: Path):
    needed = [
        data_dir / "train" / "REAL", data_dir / "train" / "FAKE",
        data_dir / "test" / "REAL", data_dir / "test" / "FAKE",
    ]
    missing = [p for p in needed if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing CIFAKE folders\n" + "\n".join(str(m) for m in missing))

def sample_paths(paths: list[Path], max_count: int | None, seed: int) -> list[Path]:
    if max_count is None or max_count >= len(paths):
        return paths
    rng = random.Random(seed)
    return sorted(rng.sample(paths, k=max_count))

def collect_split_samples(data_dir: Path, split: str, max_each: int | None, seed: int) -> list[Sample]:
    samples = []
    for cls, label in LABEL_TO_INDEX.items():
        class_dir = data_dir / split / cls
        paths = list_image_files(class_dir)
        paths = sample_paths(paths, max_each, seed)
        samples.extend(Sample(p, label) for p in paths)
    return samples

def split_train_val(samples: list[Sample], val_size: float = 0.2, seed: int = 42):
    y = [s.label for s in samples]
    idx = list(range(len(samples)))
    tr_idx, va_idx = train_test_split(idx, test_size=val_size, random_state=seed, shuffle=True, stratify=y)
    return [samples[i] for i in tr_idx], [samples[i] for i in va_idx]

def compute_accuracy(y_true, y_pred):
    return float(accuracy_score(y_true, y_pred))

def compute_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="binary"))

def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


## 4) Dataset validation and sample collection

**What this does:** validates CIFAKE folder structure, collects REAL/FAKE paths, and builds train/val/test splits.

**Why needed:** guarantees consistent labels and fair evaluation splits before training.

**Expected output:** mode summary, expected per-class counts, actual sample counts, and DataLoader batch counts.


In [ ]:
class CIFAKEDataset(Dataset):
    def __init__(self, samples: list[Sample], img_size: int = 32, corruption_fn: Callable | None = None):
        self.samples = samples
        self.corruption_fn = corruption_fn
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample.image_path).convert("RGB")
        if self.corruption_fn is not None:
            image = self.corruption_fn(image)
        return self.transform(image), sample.label

set_seed(SEED)
ensure_dir(OUTPUT_DIR)
ensure_dir(FIG_DIR)
validate_cifake_dirs(DATA_DIR)



full_expected_train_each = 50_000 if (not FAST_DEV_RUN and MAX_TRAIN_EACH is None) else MAX_TRAIN_EACH
full_expected_test_each = 10_000 if (not FAST_DEV_RUN and MAX_TEST_EACH is None) else MAX_TEST_EACH
expected_train_total = None if full_expected_train_each is None else full_expected_train_each * 2
expected_test_total = None if full_expected_test_each is None else full_expected_test_each * 2

print("=== Mode Summary ===")
print(f"FAST_DEV_RUN: {FAST_DEV_RUN}")
print(f"MAX_TRAIN_EACH: {MAX_TRAIN_EACH}")
print(f"MAX_TEST_EACH: {MAX_TEST_EACH}")
print(f"Expected train samples total: {expected_train_total}")
print(f"Expected test samples total: {expected_test_total}")

train_samples = collect_split_samples(DATA_DIR, "train", MAX_TRAIN_EACH, SEED)
test_samples = collect_split_samples(DATA_DIR, "test", MAX_TEST_EACH, SEED)
train_split, val_split = split_train_val(train_samples, val_size=0.2, seed=SEED)

print("Counts:")
print("train REAL:", sum(s.label==0 for s in train_samples))
print("train FAKE:", sum(s.label==1 for s in train_samples))
print("test REAL:", sum(s.label==0 for s in test_samples))
print("test FAKE:", sum(s.label==1 for s in test_samples))
print(f"split sizes -> train: {len(train_split)}, val: {len(val_split)}, test: {len(test_samples)}")

train_loader = DataLoader(CIFAKEDataset(train_split, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(CIFAKEDataset(val_split, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(CIFAKEDataset(test_samples, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"batch counts -> train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

if FAST_DEV_RUN:
    print("Running FAST TEST MODE: results are only for pipeline validation, not final model performance.")
else:
    print("Running FULL TRAINING MODE: this will train on the full CIFAKE dataset.")



## 5) ResNet-18 model definition

**What this does:** builds torchvision ResNet-18 adapted for CIFAKE 32x32 images (small conv1, no maxpool, 2-class head).

**Why needed:** default ImageNet stem is suboptimal for 32x32 inputs.

**Expected output:** model summary and selected device placement readiness.


In [ ]:
def build_resnet18_cifake(num_classes: int = 2):
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_resnet18_cifake(2).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model.__class__.__name__, "ready on", DEVICE)


## 6) Training and evaluation helper functions

**What this does:** defines reusable loops for training, validation, metrics, checkpointing, and plotting helpers.

**Why needed:** keeps the workflow modular and consistent across clean/robustness runs.

**Expected output:** function definitions only (no heavy computation yet).


In [ ]:
def run_eval(model, loader, criterion=None):
    model.eval()
    y_true, y_pred = [], []
    losses = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            if criterion is not None:
                losses.append(criterion(logits, labels).item())
            preds = torch.argmax(logits, dim=1)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
    out = {"accuracy": compute_accuracy(y_true, y_pred), "f1": compute_f1(y_true, y_pred), "y_true": y_true, "y_pred": y_pred}
    if criterion is not None:
        out["loss"] = float(sum(losses) / max(1, len(losses)))
    return out

def train_model(model, train_loader, val_loader, epochs):
    best_f1 = -1.0
    ckpt_path = OUTPUT_DIR / "best_resnet18.pth"
    rows = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        y_true, y_pred = [], []
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(preds.detach().cpu().tolist())

        tr_loss = float(running_loss / max(1, len(train_loader)))
        tr_acc, tr_f1 = compute_accuracy(y_true, y_pred), compute_f1(y_true, y_pred)
        val = run_eval(model, val_loader, criterion)
        rows.append({"epoch": epoch, "train_loss": tr_loss, "train_accuracy": tr_acc, "train_f1": tr_f1, "val_loss": val["loss"], "val_accuracy": val["accuracy"], "val_f1": val["f1"]})
        print(rows[-1])
        if val["f1"] > best_f1:
            best_f1 = val["f1"]
            torch.save(model.state_dict(), ckpt_path)
            print(f"Saved improved checkpoint: {ckpt_path}")

    log_df = pd.DataFrame(rows)
    save_csv(log_df, OUTPUT_DIR / "training_log.csv")
    return log_df


## 7) Training loop

**What this does:** trains ResNet-18 and saves the best checkpoint by validation F1.

**Why needed:** model selection must be based on validation data (not test data).

**Expected output:** per-epoch logs and `best_resnet18.pth`.

**Mode interpretation note:** if logs show `Epoch 1/1` and very few batches, that is FAST TEST MODE. In FULL TRAINING MODE, you should see multiple epochs and many more batches.


In [ ]:
if RUN_TRAINING:
    training_log_df = train_model(model, train_loader, val_loader, EPOCHS)
else:
    print("RUN_TRAINING=False: training skipped.")


## 8) Clean test evaluation

**What this does:** evaluates the best model on the clean CIFAKE test split.

**Why needed:** establishes baseline test performance before corruptions.

**Expected output:** clean metrics CSV and confusion matrix PNG.


In [ ]:
def clean_evaluation(model):
    ckpt = OUTPUT_DIR / "best_resnet18.pth"
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint missing: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.to(DEVICE)
    result = run_eval(model, test_loader, criterion=None)

    clean_df = pd.DataFrame([{"accuracy": result["accuracy"], "f1": result["f1"]}])
    save_csv(clean_df, OUTPUT_DIR / "resnet18_clean_results.csv")

    report = classification_report(result["y_true"], result["y_pred"], target_names=["REAL","FAKE"], output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    report_df.to_csv(OUTPUT_DIR / "resnet18_classification_report.csv")

    cm = confusion_matrix(result["y_true"], result["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["REAL", "FAKE"])
    fig, ax = plt.subplots(figsize=(5,4))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("ResNet-18 Confusion Matrix (Clean Test)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "resnet18_confusion_matrix.png", dpi=160)
    plt.close(fig)
    return result, report_df

if RUN_CLEAN_EVAL:
    clean_result, clean_report_df = clean_evaluation(model)
    print(clean_result)
else:
    print("RUN_CLEAN_EVAL=False: clean evaluation skipped.")


## 9) Robustness evaluation

**What this does:** evaluates JPEG (75/50/25), Gaussian blur (1/2/3), and Gaussian noise (0.05/0.10/0.20).

**Why needed:** measures performance drop under realistic degradations using the same test images.

**Expected output:** `resnet18_robustness_results.csv` with accuracy drops from clean accuracy.


In [ ]:
def apply_jpeg_compression(image: Image.Image, quality: int) -> Image.Image:
    buffer = io.BytesIO()
    image.convert("RGB").save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

def apply_gaussian_blur(image: Image.Image, sigma: float) -> Image.Image:
    return image.filter(ImageFilter.GaussianBlur(radius=sigma))

def apply_gaussian_noise(image: Image.Image, sigma: float) -> Image.Image:
    arr = np.asarray(image).astype(np.float32) / 255.0
    noise = np.random.normal(0.0, sigma, arr.shape).astype(np.float32)
    out = np.clip(arr + noise, 0.0, 1.0)
    return Image.fromarray((out * 255.0).astype(np.uint8))

def build_test_loader(corruption_fn=None):
    ds = CIFAKEDataset(test_samples, IMG_SIZE, corruption_fn=corruption_fn)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

def robustness_evaluation(model):
    ckpt = OUTPUT_DIR / "best_resnet18.pth"
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint missing: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.to(DEVICE)

    clean = run_eval(model, build_test_loader())
    clean_acc = clean["accuracy"]
    rows = [{"model":"resnet18", "corruption":"clean", "level":"none", "clean_acc":clean_acc, "corrupted_acc":clean_acc, "accuracy_drop":0.0, "f1":clean["f1"]}]

    for q in JPEG_QUALITIES:
        out = run_eval(model, build_test_loader(lambda im, q=q: apply_jpeg_compression(im, q)))
        rows.append({"model":"resnet18", "corruption":"jpeg", "level":q, "clean_acc":clean_acc, "corrupted_acc":out["accuracy"], "accuracy_drop":clean_acc-out["accuracy"], "f1":out["f1"]})
    for s in GAUSSIAN_BLUR_SIGMAS:
        out = run_eval(model, build_test_loader(lambda im, s=s: apply_gaussian_blur(im, s)))
        rows.append({"model":"resnet18", "corruption":"gaussian_blur", "level":s, "clean_acc":clean_acc, "corrupted_acc":out["accuracy"], "accuracy_drop":clean_acc-out["accuracy"], "f1":out["f1"]})
    for s in GAUSSIAN_NOISE_SIGMAS:
        out = run_eval(model, build_test_loader(lambda im, s=s: apply_gaussian_noise(im, s)))
        rows.append({"model":"resnet18", "corruption":"gaussian_noise", "level":s, "clean_acc":clean_acc, "corrupted_acc":out["accuracy"], "accuracy_drop":clean_acc-out["accuracy"], "f1":out["f1"]})

    df = pd.DataFrame(rows)
    save_csv(df, OUTPUT_DIR / "resnet18_robustness_results.csv")
    return df

if RUN_ROBUSTNESS_EVAL:
    robustness_df = robustness_evaluation(model)
    display(robustness_df)
else:
    print("RUN_ROBUSTNESS_EVAL=False: robustness evaluation skipped.")


## 10) Visualization generation

**What this does:** creates training, robustness, and corruption/inference figure outputs.

**Why needed:** enables report-ready interpretation of model behavior.

**Expected output:** PNG figures under `outputs/resnet18_standalone/figures/`.


In [ ]:
def plot_training_curves():
    log_path = OUTPUT_DIR / "training_log.csv"
    if not log_path.exists():
        print("No training_log.csv found.")
        return
    df = pd.read_csv(log_path)
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="Train Loss")
    axes[0].plot(df["epoch"], df["val_loss"], marker="o", label="Val Loss")
    axes[0].set_title("Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(df["epoch"], df["train_accuracy"], marker="o", label="Train Accuracy")
    axes[1].plot(df["epoch"], df["val_accuracy"], marker="o", label="Val Accuracy")
    axes[1].set_title("Accuracy Curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    fig.suptitle("ResNet-18 Training Curves", fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "resnet18_training_curves.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

def create_visualizations():
    plot_training_curves()

    clean_path = OUTPUT_DIR / "resnet18_clean_results.csv"
    robust_path = OUTPUT_DIR / "resnet18_robustness_results.csv"

    if clean_path.exists():
        clean_df = pd.read_csv(clean_path)
        acc = float(clean_df.loc[0, "accuracy"])
        f1 = float(clean_df.loc[0, "f1"])

        plt.style.use("seaborn-v0_8-whitegrid")
        fig, ax = plt.subplots(figsize=(6, 4))
        metrics = ["Accuracy", "F1"]
        values = [acc, f1]
        bars = ax.bar(metrics, values, color=["#1f77b4", "#2ca02c"])
        ax.set_ylim(0, 1.0)
        ax.set_title("ResNet-18 Clean Evaluation Summary")
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.4f}", ha="center")
        fig.tight_layout()
        fig.savefig(FIG_DIR / "resnet18_clean_performance_summary.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

    if robust_path.exists():
        robust_df = pd.read_csv(robust_path)
        clean_acc = float(robust_df[robust_df["corruption"]=="clean"].iloc[0]["acc"])
        mappings = [
            ("jpeg", "JPEG Compression", [75, 50, 25]),
            ("gaussian_blur", "Gaussian Blur", [1, 2, 3]),
            ("gaussian_noise", "Gaussian Noise", [0.05, 0.10, 0.20]),
        ]

        plt.style.use("seaborn-v0_8-whitegrid")
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
        for ax, (name, title, level_order) in zip(axes, mappings):
            sub = robust_df[robust_df["corruption"] == name].copy()
            sub["level"] = pd.to_numeric(sub["level"], errors="coerce")
            sub = sub.set_index("level").reindex(level_order).reset_index()
            ax.plot(sub["level"], sub["acc"], marker="o", linewidth=2)
            ax.axhline(clean_acc, linestyle="--", color="gray", label="Clean Acc")
            ax.set_title(title)
            ax.set_xlabel("Corruption Level")
            ax.grid(alpha=0.3)
        axes[0].set_ylabel("Accuracy")
        axes[2].legend(loc="best")
        fig.suptitle("ResNet-18 Robustness Curves", fontweight="bold")
        fig.tight_layout()
        fig.savefig(FIG_DIR / "resnet18_robustness_curves.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

        avg = robust_df[robust_df["corruption"] != "clean"].groupby("corruption", as_index=False)["drop"].mean()
        avg["corruption_label"] = avg["corruption"].map({"jpeg": "JPEG", "gaussian_blur": "Gaussian Blur", "gaussian_noise": "Gaussian Noise"})
        ordered = avg.sort_values("drop", ascending=False)

        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.bar(ordered["corruption_label"], ordered["drop"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
        ax.set_title("Average Accuracy Drop by Corruption Type")
        ax.set_ylabel("Average Drop (Clean Acc - Corrupted Acc)")
        ax.set_xlabel("Corruption Type")
        fig.tight_layout()
        fig.savefig(FIG_DIR / "resnet18_avg_drop_bar.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(7, 2.5))
        heat_df = ordered[["corruption_label", "drop"]].set_index("corruption_label").T
        im = ax.imshow(heat_df.values, aspect="auto", cmap="Reds")
        ax.set_xticks(range(len(heat_df.columns)))
        ax.set_xticklabels(heat_df.columns)
        ax.set_yticks([0])
        ax.set_yticklabels(["Avg Drop"])
        for i, col in enumerate(heat_df.columns):
            ax.text(i, 0, f"{heat_df.iloc[0, i]:.4f}", ha="center", va="center", color="black", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title("Average Drop Heatmap")
        fig.tight_layout()
        fig.savefig(FIG_DIR / "resnet18_avg_drop_heatmap.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

    cm_path = OUTPUT_DIR / "resnet18_confusion_matrix.png"
    if cm_path.exists():
        display(Image.open(cm_path))

if RUN_VISUALIZATION:
    create_visualizations()
    print(f"Saved visualization outputs in: {FIG_DIR}")


## 11) Inference demo

**What this does:** runs sample-level predictions with confidence scores on selected images.

**Why needed:** provides qualitative sanity checks beyond aggregate metrics.

**Expected output:** printed predictions and saved inference example figures.


In [ ]:
def predict_with_confidence(model, pil_image: Image.Image):
    model.eval()
    t = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
    x = t(pil_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[pred])

def inference_demo(model, n_each=4):
    ckpt = OUTPUT_DIR / "best_resnet18.pth"
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint missing: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.to(DEVICE)

    rng = random.Random(SEED)
    real_paths = [s.image_path for s in test_samples if s.label==0]
    fake_paths = [s.image_path for s in test_samples if s.label==1]
    chosen = rng.sample(real_paths, min(n_each, len(real_paths))) + rng.sample(fake_paths, min(n_each, len(fake_paths)))

    fig, axes = plt.subplots(2, n_each, figsize=(3*n_each, 6))
    for i, p in enumerate(chosen):
        img = Image.open(p).convert("RGB")
        pred, conf = predict_with_confidence(model, img)
        r, c = divmod(i, n_each)
        axes[r, c].imshow(img); axes[r, c].axis("off")
        axes[r, c].set_title(f"GT:{p.parent.name} | Pred:{INDEX_TO_LABEL[pred]}
Conf:{conf:.3f}", fontsize=9)
    fig.suptitle("ResNet-18 Practical Inference on CIFAKE Test Images", fontsize=14)
    fig.tight_layout(); fig.savefig(FIG_DIR / "inference_examples.png", dpi=200); plt.close(fig)

    real = Image.open(real_paths[0]).convert("RGB")
    fake = Image.open(fake_paths[0]).convert("RGB")
    conditions = [
        ("Original", lambda x: x),
        ("JPEG q=25", lambda x: apply_jpeg_compression(x, 25)),
        ("Blur σ=3", lambda x: apply_gaussian_blur(x, 3)),
        ("Noise σ=0.2", lambda x: apply_gaussian_noise(x, 0.2)),
    ]
    fig, axes = plt.subplots(2, len(conditions), figsize=(4*len(conditions), 8))
    for r, (gt, base) in enumerate([("REAL", real), ("FAKE", fake)]):
        for c, (name, fn) in enumerate(conditions):
            img = fn(base)
            pred, conf = predict_with_confidence(model, img)
            axes[r,c].imshow(img); axes[r,c].axis("off")
            axes[r,c].set_title(f"{name}
GT:{gt} Pred:{INDEX_TO_LABEL[pred]}
Conf:{conf:.3f}", fontsize=9)
    fig.suptitle("Corruption Sensitivity Demo (REAL and FAKE)", fontsize=14)
    fig.tight_layout(); fig.savefig(FIG_DIR / "corrupted_inference_examples.png", dpi=200); plt.close(fig)

if RUN_INFERENCE_DEMO:
    inference_demo(model)
else:
    print("RUN_INFERENCE_DEMO=False: inference demo skipped.")


## 12) Output summary table

**What this does:** aggregates key clean/robustness metrics into a final summary CSV.

**Why needed:** provides one compact artifact for report writing and comparison.

**Expected output:** `resnet18_final_summary_table.csv`.


In [ ]:
# Build final summary table and save outputs
clean_csv = OUTPUT_DIR / "resnet18_clean_results.csv"
robust_csv = OUTPUT_DIR / "resnet18_robustness_results.csv"
summary_csv = OUTPUT_DIR / "resnet18_final_summary_table.csv"

if clean_csv.exists() and robust_csv.exists():
    clean_df = pd.read_csv(clean_csv)
    robust_df = pd.read_csv(robust_csv)
    avg = robust_df[robust_df["corruption"] != "clean"].groupby("corruption", as_index=False)["drop"].mean()
    avg_map = {r["corruption"]: float(r["drop"]) for _, r in avg.iterrows()}
    final = pd.DataFrame([{
        "clean_accuracy": float(clean_df.loc[0, "accuracy"]),
        "clean_f1": float(clean_df.loc[0, "f1"]),
        "avg_drop_jpeg": avg_map.get("jpeg", np.nan),
        "avg_drop_gaussian_blur": avg_map.get("gaussian_blur", np.nan),
        "avg_drop_gaussian_noise": avg_map.get("gaussian_noise", np.nan),
    }])
    final.to_csv(summary_csv, index=False)
    display(final)
    print(f"Saved: {summary_csv}")
else:
    print("Skipping summary table: clean/robustness CSV not found yet.")


In [ ]:
# Output inspection
!find outputs/resnet18_standalone -maxdepth 3 -type f


## 13) Zip backup and download

**What this does:** archives notebook outputs into a single zip.

**Why needed:** simplifies downloading all artifacts from Kaggle/Codex runtime.

**Expected output:** `resnet18_standalone_outputs.zip`.


In [ ]:
zip_path = Path("resnet18_standalone_outputs.zip")
if OUTPUT_DIR.exists():
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(zip_path.with_suffix(""), "zip", OUTPUT_DIR.parent, OUTPUT_DIR.name)
    print(f"Created: {zip_path.resolve()}")
    print("In Kaggle, open the right-side Output panel to download the zip file.")
else:
    print("OUTPUT_DIR does not exist yet.")


## 14) Expected final outputs

After a successful FULL TRAINING MODE run, you should have:

- `best_resnet18.pth`
- `training_log.csv`
- `resnet18_clean_results.csv`
- `resnet18_robustness_results.csv`
- `resnet18_confusion_matrix.png`
- `figures/` training curves
- `figures/` robustness curves
- `figures/` average drop heatmap
- `figures/` corruption examples
- `figures/` inference examples
- `resnet18_final_summary_table.csv`
- `resnet18_standalone_outputs.zip`
